# AffectScore — LoRA Training

Train the rank-sweep and ablation model variants. Run `AffectScore - Gates.ipynb` first
and confirm all three gates pass before proceeding.

**Requirements**
- Colab Pro+ (A100 GPU, ~6 h per 50-epoch run)
- `affectscore-colab.zip` uploaded to `MyDrive/affectscore/affectscore-colab.zip`
- Hugging Face token with **write** access (for checkpoint upload)
- Zenodo record ID filled in below (training WAVs)

In [ ]:
from google.colab import drive, userdata
import os, subprocess

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/affectscore"
REPO  = "/content/affectscore"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/LeeTgk/affectscore.git", REPO], check=True)

print(f"Repo ready: {REPO}")


In [ ]:
!bash /content/affectscore/training/colab_setup.sh


## Download dataset from Zenodo

Downloads the 1,571-clip preprocessed WAV dataset and the curated training manifest.
Skips automatically if already present on Drive.

In [ ]:
import os, subprocess, shutil, tarfile, glob

ZENODO_RECORD = "21830658"   # https://zenodo.org/record/21830658
PREPROCESSED  = f"{DRIVE}/preprocessed_unfiltered"

if not os.path.isdir(PREPROCESSED):
    print("Downloading dataset from Zenodo...")
    subprocess.run(["pip", "install", "zenodo-get", "-q"], check=True)
    subprocess.run(["zenodo_get", ZENODO_RECORD, "-o", DRIVE], check=True)

    # Extract archive if Zenodo record ships a tar.gz
    for archive in glob.glob(f"{DRIVE}/*.tar.gz"):
        print(f"Extracting {archive}...")
        with tarfile.open(archive) as t:
            t.extractall(DRIVE)
    print("Download complete.")
else:
    print(f"Dataset already present: {PREPROCESSED}")

# Copy dataset manifests into repo data/ so scripts can find them
os.makedirs(f"{REPO}/data", exist_ok=True)
for fname in ["training_set_clean_clap.json", "held_out_set.json"]:
    _src = f"{DRIVE}/{fname}"
    _dst = f"{REPO}/data/{fname}"
    if os.path.exists(_src) and not os.path.exists(_dst):
        shutil.copy2(_src, _dst)
        print(f"Copied {fname} -> {REPO}/data/")



In [ ]:
%cd /content


## Rank sweep — r=16, r=32, r=64

Three independently trained models. r=32 is the primary model used in all paper results.
r=16 and r=64 participate in the rank ablation (Table 2).

In [ ]:
!python -u affectscore/training/train_lora.py \
  --rank 16 \
  --manifest affectscore/data/training_set_clean_clap.json \
  --epochs 50 --lr 1e-4 --batch-size 8 --max-quadrant-weight 5.0 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

In [ ]:
!python -u affectscore/training/train_lora.py \
  --rank 32 \
  --manifest affectscore/data/training_set_clean_clap.json \
  --epochs 50 --lr 1e-4 --batch-size 8 --max-quadrant-weight 5.0 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

In [ ]:
!python -u affectscore/training/train_lora.py \
  --rank 64 \
  --manifest affectscore/data/training_set_clean_clap.json \
  --epochs 50 --lr 1e-4 --batch-size 8 --max-quadrant-weight 5.0 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

## Ablation variants

Each is a **separately trained** model — inputs are never zeroed at inference time.

| Variant | What is removed |
|---------|----------------|
| `no-affect` | Layer 2 engagement conditioning (choice latency, dwell deviation, interaction rate) |
| `no-style`  | CLAP auxiliary loss (λ = 0); style alignment term absent from training objective |

In [ ]:
!python -u affectscore/training/train_lora.py \
  --rank 32 --lambda_clap 0.1 --ablation no-affect \
  --manifest affectscore/data/training_set_clean_clap.json \
  --epochs 50 --lr 1e-4 --batch-size 8 --max-quadrant-weight 5.0 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

In [ ]:
!python -u affectscore/training/train_lora.py \
  --rank 32 --lambda_clap 0.0 --ablation no-style \
  --manifest affectscore/data/training_set_clean_clap.json \
  --epochs 50 --lr 1e-4 --batch-size 8 --max-quadrant-weight 5.0 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

## SAO comparison

Spectrum-Aware Optimisation baseline (Lv et al. 2024). Provides an alternative
fine-tuning strategy trained under identical conditions for comparison.

In [ ]:
!pip install git+https://github.com/NeuralNotW0rk/LoRAW.git stable-audio-tools -q


In [ ]:
!python -u affectscore/training/train_sao.py \
  --rank 32 \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

## Upload adapter checkpoints to Hugging Face

Set `HF_USERNAME` to your Hugging Face username. Update `runs` to match the
checkpoint directory names created during training (timestamped by train_lora.py).

In [ ]:
from huggingface_hub import HfApi
import os, re

api          = HfApi()
token        = os.environ["HF_TOKEN"]
HF_USERNAME  = "HiiragiLee"

# Update run IDs to match your checkpoint directory names
runs = [
    "affectscore-ace-step-r16-20260629",
    "affectscore-ace-step-r32-20260629",
    "affectscore-ace-step-r64-20260629",
    "affectscore-ace-step-r32-20260629-no-affect",
    "affectscore-ace-step-r32-20260629-no-style",
    "affectscore-ace-step-r32-20260629-no-lora",
]

for run_id in runs:
    local_path = f"{DRIVE}/checkpoints/{run_id}/final_adapter"
    readme = os.path.join(local_path, "README.md")
    if os.path.exists(readme):
        with open(readme) as f:
            content = f.read()
        content = re.sub(r"base_model:\s*.+", "base_model: ACE-Step/ACE-Step-v1-3.5B", content)
        with open(readme, "w") as f:
            f.write(content)
    repo_id = f"{HF_USERNAME}/{run_id}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path=local_path, repo_id=repo_id, token=token)
    print(f"Uploaded: {repo_id}")
